In [10]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
class Node:
    def __init__(self,feature=None,threshold=None,right=None,left=None,value=None):
        self.feature = feature
        self.threshold = threshold
        self.right = right
        self.left = left
        self.value = value
        
    def is_leaf_node(self):
#         if self.value != None:
#             return True
#         else:
#             return False
        return self.value is not None
        
class DecisionTree:
    def __init__(self):
        self.root = None
    
    def fit(self,X,y):
        self.root = self.build_tree(X,y)
        
    def build_tree(self,X,y):
        if len(np.unique(y))==1:
            node = Node(value=y[0])
            return node
        
        feature,threshold,gain=self.best_split(X,y)
        
        if gain ==0:
            classes,count = np.unique(y,return_counts=True)
            majority_index = np.argmax(count)
            majority_class = classes[majority_index]
            node = Node(value=majority_class)
            return node
        
        left_X,left_y,right_X,right_y = self.split(feature,threshold,X,y)
        left_node = self.build_tree(left_X,left_y)
        right_node = self.build_tree(right_X,right_y)
        node = Node(feature=feature,threshold=threshold,left=left_node,right=right_node)
        return node
        
    def split(self,feature,threshold,X,y):
        left_indices = []
        right_indices = []
        
        for i in range(len(X)):
            if X[i][feature] <= threshold:
                left_indices.append(i)
            else:
                right_indices.append(i)
        
        left_X = X[left_indices]
        left_y = y[left_indices]
        right_X = X[right_indices]
        right_y = y[right_indices]
        
        return left_X,left_y,right_X,right_y
    
    
    
    
    def best_split(self,X,y):
        best_gain = 0
        best_feature = None
        best_threshold = None
        
        for feature in range(X.shape[1]):
            feature_values = X[:,feature]
            unique_values = np.unique(feature_values)
            for i in range(len(unique_values)-1):
                threshold_value = (unique_values[i] + unique_values[i+1])/2
                left_X,left_y,right_X,right_y = self.split(feature,threshold_value,X,y)
                gain = information_gain(y, left_y, right_y)
                
                if best_gain < gain:
                    best_gain = gain
                    best_feature = feature
                    best_threshold = threshold_value
                    
        return best_feature, best_threshold, best_gain
    
    def predict_one(self,x):
        node = self.root
        
        while not node.is_leaf_node():
            if x[node.feature]<= node.threshold:
                node = node.left
            else:
                node = node.right
        return node.value
    
    def accuracy(self,X_test,y_test):
        correct_count = 0
        for i in range(len(X_test)):
            pred = self.predict_one(X_test[i])
            if pred == y_test[i]:
                correct_count+=1
        accuracy = (correct_count/len(y_test))*100
        return accuracy
            
                
def entropy(y):
    classes,count = np.unique(y,return_counts=True)
    probabilities = count/len(y)
    entropy_value = -np.sum(probabilities*np.log2(probabilities))
    return entropy_value

def information_gain(y, left_y, right_y):
    parent_entropy = entropy(y)
    left_entropy = entropy(left_y)
    right_entropy = entropy(right_y)
    w_left = len(left_y)/len(y)
    w_right = len(right_y)/len(y)
    info_gain = parent_entropy-((left_entropy*w_left)+(right_entropy*w_right))
    return info_gain

In [11]:
iris = load_iris()
X = iris.data
y = iris.target
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
tree = DecisionTree()
tree.fit(X_train, y_train)

prediction = tree.predict_one(X_test[5])
print(prediction)
accuracy = tree.accuracy(X_test,y_test)
print(accuracy)

0
100.0
